# Day 25 - Autoencoders for Anomaly Detection

This notebook implements an Autoencoder from scratch using TensorFlow/Keras for anomaly detection. Each section contains explanations followed by executable code.

## Import Libraries

Import all required libraries for data generation, preprocessing, deep learning, evaluation, and visualization.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Create Normal and Anomaly Data

Generate synthetic normal samples and anomaly samples. Train the autoencoder only on normal data.

In [ ]:
np.random.seed(42)

X_normal = np.random.randn(1000,10)*0.5 + 2
X_anomaly = np.random.randn(50,10)*2 - 3

scaler = StandardScaler()

X_train, X_test_normal = train_test_split(X_normal,test_size=0.2,random_state=42)

X_train = scaler.fit_transform(X_train)
X_test_normal = scaler.transform(X_test_normal)
X_anomaly = scaler.transform(X_anomaly)

print("Training:",len(X_train))
print("Normal Test:",len(X_test_normal))
print("Anomalies:",len(X_anomaly))

## Build the Autoencoder

Create encoder, bottleneck, decoder, and compile the model using Mean Squared Error loss.

In [ ]:
input_dim = X_train.shape[1]

inputs = keras.Input(shape=(input_dim,))
x = layers.Dense(8,activation="relu")(inputs)
latent = layers.Dense(4,activation="relu",name="bottleneck")(x)
x = layers.Dense(8,activation="relu")(latent)
outputs = layers.Dense(input_dim,activation="linear")(x)

autoencoder = keras.Model(inputs,outputs)
encoder = keras.Model(inputs,latent)

autoencoder.compile(optimizer="adam",loss="mse")
autoencoder.summary()

## Train the Autoencoder

Train using only normal data so the network learns normal patterns.

In [ ]:
history = autoencoder.fit(
    X_train,
    X_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

plt.figure(figsize=(8,4))
plt.plot(history.history["loss"],label="Train")
plt.plot(history.history["val_loss"],label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Curve")
plt.legend()
plt.show()

## Reconstruction Error

Calculate reconstruction errors for normal and anomaly samples.

In [ ]:
recon_normal = autoencoder.predict(X_test_normal,verbose=0)
recon_anomaly = autoencoder.predict(X_anomaly,verbose=0)

mse_normal = np.mean((X_test_normal-recon_normal)**2,axis=1)
mse_anomaly = np.mean((X_anomaly-recon_anomaly)**2,axis=1)

print("Average Normal Error:",mse_normal.mean())
print("Average Anomaly Error:",mse_anomaly.mean())

## Threshold-Based Detection

Use Mean + 2×Standard Deviation of normal reconstruction errors as the anomaly threshold.

In [ ]:
threshold = mse_normal.mean() + 2*mse_normal.std()
print("Threshold:",threshold)

normal_pred = mse_normal > threshold
anomaly_pred = mse_anomaly > threshold

print("Normal flagged:",normal_pred.sum(),"/",len(normal_pred))
print("Anomalies detected:",anomaly_pred.sum(),"/",len(anomaly_pred))

## Visualizations

Visualize reconstruction error distributions, latent space, and original vs reconstructed sample.

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(12,10))

ax[0,0].hist(mse_normal,bins=25,alpha=0.7,label="Normal")
ax[0,0].hist(mse_anomaly,bins=25,alpha=0.7,label="Anomaly")
ax[0,0].axvline(threshold,linestyle="--")
ax[0,0].legend()
ax[0,0].set_title("Reconstruction Error")

latent_normal=encoder.predict(X_test_normal,verbose=0)
latent_anomaly=encoder.predict(X_anomaly,verbose=0)

ax[0,1].scatter(latent_normal[:,0],latent_normal[:,1],s=15,label="Normal")
ax[0,1].scatter(latent_anomaly[:,0],latent_anomaly[:,1],marker="x",label="Anomaly")
ax[0,1].legend()
ax[0,1].set_title("Latent Space")

ax[1,0].plot(X_test_normal[0],label="Original")
ax[1,0].plot(recon_normal[0],label="Reconstructed")
ax[1,0].legend()
ax[1,0].set_title("Normal Reconstruction")

y_true=np.concatenate([np.zeros(len(mse_normal)),np.ones(len(mse_anomaly))])
scores=np.concatenate([mse_normal,mse_anomaly])
auc=roc_auc_score(y_true,scores)
ax[1,1].text(0.1,0.5,f"AUC = {auc:.4f}",fontsize=16)
ax[1,1].axis("off")

plt.tight_layout()
plt.show()

# Summary

- Autoencoders learn only **normal patterns**.
- High reconstruction error indicates potential anomalies.
- Suitable for images, signals, sensor data, and time series.
- Thresholding reconstruction error enables anomaly detection.
